# Surgical Vision — run all experiments on Colab

1. Runtime → Change runtime type → **GPU** (T4 or better).
2. Course data must be in Google Drive under `MyDrive/BN5211/` (`Dataset/`, `dataset-full/`, `RARP_1FPS/`).
3. Results go to `MyDrive/BN5211/runs/`. Finished runs are skipped, so after a disconnect just run the setup cells again and re-run the project cell that was interrupted.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d repo && git -C repo pull --ff-only || git clone https://github.com/weirdquantum/surgical-vision-moco.git repo
%cd /content/repo
!pip install -q pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/BN5211'
CACHE = f'{DRIVE}/cache/data_cache.tar'   # Dataset + dataset-full + resized RARP frames (~140 MB)
!mkdir -p /content/data {DRIVE}/runs {DRIVE}/cache
!ln -sfn /content/data data
!ln -sfn {DRIVE}/runs runs
import os
if os.path.exists(CACHE):
    # Later sessions: one large file copies in seconds (Drive is slow per file).
    !tar -xf {CACHE} -C /content/data
else:
    # First session: copy thousands of files from Drive (5-15 min), resize, then cache.
    !rsync -a --info=progress2 {DRIVE}/Dataset {DRIVE}/dataset-full {DRIVE}/RARP_1FPS /content/data/
    !python -m action_recognition.prepare --data-root data --workers 8
    !tar -cf {CACHE} -C /content/data Dataset dataset-full RARP_1FPS_288x512
!ls data && du -sh data/*

In [ ]:
!python -m pytest -q tests

## Project 1 — instrument classification

In [ ]:
!bash scripts/run_all.sh core p1 2>&1 | tee -a runs/colab_p1.log

## Project 2 — action recognition (longest)

In [ ]:
!bash scripts/run_all.sh core p2 2>&1 | tee -a runs/colab_p2.log

## Project 3 — MoCo pretraining + fine-tuning

In [ ]:
!bash scripts/run_all.sh core p3 2>&1 | tee -a runs/colab_p3.log

## Extra runs
Longer schedule for the from-scratch CNN (~30 min) and leakage-free 4-fold cross-validation over whole surgeries for action recognition (~20-30 min).

In [ ]:
!python -m instrument_classification.train --config instrument_classification/configs/convnet_scratch_300ep.yaml 2>&1 | tee -a runs/colab_extra.log
!python -m action_recognition.train --config action_recognition/configs/convnext_tiny_surgery_cv.yaml 2>&1 | tee -a runs/colab_extra.log
!cat runs/instrument_classification/convnet_scratch_300ep/summary.md runs/action_recognition/convnext_tiny_surgery_cv/summary.md

## Final runs
Remaining experiments (~1 h on an A100): complete C0-C5 reproduction, MS-TCN under surgery-level CV, cross-fitting ablation, 300-epoch MoCo fine-tuning. Writes `runs/ALL_RESULTS.md`. Use `label` instead of `required` to add the label-efficiency study (~2 h more).

In [ ]:
%cd /content/repo
!bash scripts/run_final.sh required 2>&1 | tee -a runs/colab_final.log

## Summaries

In [ ]:
!for f in runs/*/*/summary.md; do echo "== $f"; cat "$f"; done
!cat runs/contrastive_learning/probe/probe.json 2>/dev/null | head -50